In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import category_encoders as ce

In [ ]:
data['old'] = (data['age'] > 60).astype(int)

data['old'] = np.where(data['age'] > 60, 1, 0)

data['old'] = data['age'].apply(lambda x: 1 if x > 60 else 0)

In [ ]:
import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn import preprocessing
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================

data = pd.read_csv('heart.csv')

# ============================================
# 2. СОЗДАНИЕ ПРИЗНАКА trestbps_mean
# ============================================

# Словарь с нормами давления
norm_df = pd.DataFrame([
    ('до 20', 'Мужчина', 123),
    ('до 20', 'Женщина', 116),
    ('21-30', 'Мужчина', 126),
    ('21-30', 'Женщина', 120),
    ('31-40', 'Мужчина', 129),
    ('31-40', 'Женщина', 127),
    ('41-50', 'Мужчина', 135),
    ('41-50', 'Женщина', 137),
    ('51-60', 'Мужчина', 142),
    ('51-60', 'Женщина', 144),
    ('61+', 'Мужчина', 142),
    ('61+', 'Женщина', 159)
], columns=['age_group', 'sex_label', 'trestbps_mean'])

# Определяем возрастные группы
data['age_group'] = pd.cut(
    data['age'],
    bins=[0, 20, 30, 40, 50, 60, 200],
    labels=['до 20', '21-30', '31-40', '41-50', '51-60', '61+'],
    right=False
)

# Преобразуем пол в текстовые метки
data['sex_label'] = data['sex'].map({1: 'Мужчина', 0: 'Женщина'})

# Объединяем с таблицей норм
data = data.merge(norm_df, on=['age_group', 'sex_label'], how='left')

# Удаляем временные колонки
data = data.drop(['age_group', 'sex_label'], axis=1)

# ============================================
# 3. ONE-HOT ENCODING
# ============================================

encoder = ce.OneHotEncoder(
    cols=['cp', 'restecg', 'slope', 'ca', 'thal']
)
data_encoded = encoder.fit_transform(data)

# ============================================
# 4. МАСШТАБИРОВАНИЕ (RobustScaler)
# ============================================

# Проверяем, что все колонки числовые
print("Типы данных после One-Hot:")
print(data_encoded.dtypes.value_counts())

# Масштабируем
scaler = preprocessing.RobustScaler()
column_names = data_encoded.columns.tolist()

data_scaled = scaler.fit_transform(data_encoded)
data_scaled = pd.DataFrame(data_scaled, columns=column_names)

print(f"\nРазмер после масштабирования: {data_scaled.shape}")

# ============================================
# 5. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ
# ============================================

# Матрица корреляций
corr_matrix = data_scaled.corr()

# Визуализация тепловой карты
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Матрица корреляций признаков', fontsize=14)
plt.tight_layout()
plt.show()

# Поиск сильно коррелированных пар
threshold = 0.7
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
high_corr_pairs = corr_matrix.where(mask).stack()
high_corr_pairs = high_corr_pairs[abs(high_corr_pairs) > threshold]

print("\n=== СИЛЬНО КОРРЕЛИРОВАННЫЕ ПРИЗНАКИ ===")
print(f"Порог: |corr| > {threshold}")
print("-" * 50)

if len(high_corr_pairs) > 0:
    for (col1, col2), corr_value in high_corr_pairs.items():
        direction = 'положительная' if corr_value > 0 else 'отрицательная'
        print(f"  {col1} ↔ {col2}: {corr_value:.3f} ({direction})")
else:
    print("  Сильно коррелированных пар не найдено")

# ============================================
# 6. ВЫВОД ИНФОРМАЦИИ О ДАННЫХ
# ============================================

print("\n=== ИТОГОВАЯ ИНФОРМАЦИЯ ===")
print("-" * 50)
print(f"Исходный датасет: {data.shape}")
print(f"После One-Hot: {data_encoded.shape}")
print(f"После масштабирования: {data_scaled.shape}")
print(f"Колонок с высокой корреляцией: {len(high_corr_pairs)}")

# Показываем первые 5 строк результата
print("\n=== ПЕРВЫЕ 5 СТРОК МАСШТАБИРОВАННЫХ ДАННЫХ ===")
print(data_scaled.head())

In [ ]:
data = pd.read_csv('heart.csv')

# Создаём таблицу норм
norm_df = pd.DataFrame([
    ('до 20', 'Мужчина', 123), ('до 20', 'Женщина', 116),
    ('21-30', 'Мужчина', 126), ('21-30', 'Женщина', 120),
    ('31-40', 'Мужчина', 129), ('31-40', 'Женщина', 127),
    ('41-50', 'Мужчина', 135), ('41-50', 'Женщина', 137),
    ('51-60', 'Мужчина', 142), ('51-60', 'Женщина', 144),
    ('61+', 'Мужчина', 142), ('61+', 'Женщина', 159)
], columns=['age_group', 'sex_label', 'trestbps_mean'])

# Определяем группы
data['age_group'] = pd.cut(data['age'], 
                           bins=[0, 20, 30, 40, 50, 60, 200],
                           labels=['до 20', '21-30', '31-40', '41-50', '51-60', '61'],
                           right=False)
data['sex_label'] = data['sex'].map({1: 'Мужчина', 0: 'Женщина'})

# Объединяем (merge) — очень быстро на больших данных
data = data.merge(norm_df, on=['age_group', 'sex_label'], how='left')



encoder = ce.OneHotEncoder(cols=['cp', 'restecg', 'slope', 'ca', 'thal'])
data_onehot = encoder.fit_transform(data)


from sklearn import preprocessing
# инициализируем нормализатор RobustScaler
r_scaler = preprocessing.RobustScaler()
col_names = list(data.columns)

In [ ]:
list_of_dicts = [
 {'product': 'Product1', 'price': 1200, 'payment_type': 'Mastercard'},
 {'product': 'Product2', 'price': 3600, 'payment_type': 'Visa'},
 {'product': 'Product3', 'price': 7500, 'payment_type': 'Amex'}
]
df = pd.DataFrame(list_of_dicts)
encoder = ce.OneHotEncoder(cols=['product', 'payment_type'], use_cat_names=False)
cols = encoder.fit_transform(df[['product','payment_type']])
df = pd.concat([df, cols], axis=1) # Лучшая визуализация


# инициализируем информацию об одежде
clothing_list = [
    ['xxs', 'dress'],
    ['xxs', 'skirt'],
    ['xs', 'dress'],
    ['s', 'skirt'],
    ['m', 'dress'],
    ['l', 'shirt'],
    ['s', 'coat'],
    ['m', 'coat'],
    ['xxl', 'shirt'],
    ['l', 'dress']
]
clothing = pd.DataFrame(clothing_list, columns = ['size',  'type'])
# Кодируем размер (порядковый признак)
size_encoder = ce.OrdinalEncoder(mapping=[{
    'col': 'size',
    'mapping': {'xxs': 1, 'xs': 2, 's': 3, 'm': 4, 'l': 5, 'xxl': 6}
}])
clothing['size_code'] = size_encoder.fit_transform(clothing[['size']])


# Виды
data = pd.DataFrame({
    'city': ['Moscow', 'London', 'London', 'Kiev', 'Moscow'],
    'price': [100, 200, 150, 80, 120]
})

# 1. ONE-HOT ENCODING
encoder = ce.OneHotEncoder(cols=['city'], use_cat_names=True)
data_onehot = encoder.fit_transform(data)

# 2. ORDINAL ENCODING
encoder = ce.OrdinalEncoder(cols=['city'])
data_ordinal = encoder.fit_transform(data)

# 3. BINARY ENCODING
encoder = ce.BinaryEncoder(cols=['city'])
data_binary = encoder.fit_transform(data)

# 4. TARGET ENCODING (с учителем)
encoder = ce.TargetEncoder(cols=['city'])
data_target = encoder.fit_transform(data['city'], data['price']) 